# 2 - Préparation des données

Découpage de données

* **Entrée** : `student-por.csv` (via `outils.load_data`)
* **Sortie** : `split/` contenant `X_train`, `X_test`, `y_train`, `y_test` (données brutes, **non encodées**)

Le prétraitement (standardisation, one-hot) est défini dans `outils.py` mais sera **appris dans `3-modeles`**, à l'intérieur d'un `Pipeline`, sur le train uniquement. Cela évite toute fuite d'information vers le test.

# Importation des librairies et des données

In [21]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import outils as ot
from sklearn.model_selection import train_test_split

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
df = ot.load_data()
X = df.drop(columns=ot.TARGET)   # 32 variables, G1 et G2 comprises
y = df[ot.TARGET]
print(X.shape, y.shape)

(649, 32) (649,)


# Séparation train / test

20% des données sont réservées pour le test, qui ne sera utilisé **qu'une seule fois**, dans `4-evaluation`. Le choix des modèles se fera uniquement par validation croisée sur le `train`.

Nous gardons les 15 élèves avec `G3 = 0` dans le jeu de données : la raison de leur note est méconnue (note non attribuée ou vrai zéro), et la variante « sans zéros » peut être obtenue plus tard en filtrant `y_train > 0` dans `3-modeles`. Avec le même split, la comparaison entre les deux variantes reste équitable.

En effet, comme ces 15 cas sont rares, nous utilisons `stratify=(y == 0)` : ils sont répartis proportionnellement entre train et test, au lieu de risquer qu'ils tombent presque tous du même côté.

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=ot.SEED, stratify=(y == 0))
print(f"Train : {len(X_train)} élèves | Test : {len(X_test)} élèves")
print("Élèves avec G3 = 0 : train", (y_train == 0).sum(), "| test", (y_test == 0).sum())

Train : 519 élèves | Test : 130 élèves
Élèves avec G3 = 0 : train 12 | test 3


## `train` et `test` comparables ?

In [24]:
pd.DataFrame({"train": y_train.describe(), "test": y_test.describe()}).round(2)

,train,test
count,519.00,130.00
mean,11.98,11.61
std,3.24,3.20
min,0.00,0.00
25%,10.00,10.00
50%,12.00,11.00
75%,14.00,13.00
max,19.00,18.00


`train` et `test` sont très similaires, le jeu de données a été bien séparé.

In [25]:
diffs = {c: X_train[c].value_counts(normalize=True)
              .sub(X_test[c].value_counts(normalize=True), fill_value=0).abs().max()
         for c in ot.CATEGORICAL}
pd.Series(diffs).sort_values(ascending=False).round(2)

school        0.11
famsize       0.09
reason        0.08
nursery       0.07
higher        0.05
activities    0.05
guardian      0.05
famsup        0.04
schoolsup     0.03
address       0.02
Fjob          0.02
Mjob          0.02
romantic      0.02
paid          0.02
internet      0.01
sex           0.01
Pstatus       0.00
dtype: float64

In [26]:
pd.DataFrame({"train": X_train.school.value_counts(normalize=True),
             "test": X_test.school.value_counts(normalize=True)}).round(2)

,train,test
school,,
GP,0.67,0.56
MS,0.33,0.44


L'écart de proportion maximal entre `train` et `test` est de 11 de points de pourcentage (`school`). C'est notable mais pas alarmant.

La moyenne de jeu de `test` (11,61) est légèrement plus faible que la moyenne du jeu d'entraînement (11,98). Cela est probablement dû au fait que l'école MS est y est surreprésentée et que cette classe concentre beaucoup de `G3=0`.

# Test du préprocesseur

On vérifie que `build_preprocessor` fonctionne pour chaque scénario : appris sur le train seulement (`fit_transform`), puis appliqué au test avec `transform` sans le réapprendre.

In [27]:
for nom, feats in ot.SCENARIOS.items():
    prep = ot.build_preprocessor(feats)
    Xt = prep.fit_transform(X_train)                  # apprentissage sur le train uniquement
    Xt = Xt.toarray() if hasattr(Xt, "toarray") else Xt # on s'assure que le format soit dense
    Xte = prep.transform(X_test)                      # le test est seulement transformé
    print(f"{nom} : {len(feats)} variables -> {Xt.shape[1]} colonnes encodées | "
          f"NaN dans le train : {pd.isna(Xt).sum()} | forme du test : {Xte.shape}")

A_G1_G2 : 32 variables -> 58 colonnes encodées | NaN dans le train : 0 | forme du test : (130, 58)
B_G1 : 31 variables -> 57 colonnes encodées | NaN dans le train : 0 | forme du test : (130, 57)
C_sans_notes : 30 variables -> 56 colonnes encodées | NaN dans le train : 0 | forme du test : (130, 56)


In [28]:
# Aperçu des colonnes produites par le dernier scénario
list(prep.get_feature_names_out())[:15]

['num__age',
 'num__Medu',
 'num__Fedu',
 'num__traveltime',
 'num__studytime',
 'num__failures',
 'num__famrel',
 'num__freetime',
 'num__goout',
 'num__Dalc',
 'num__Walc',
 'num__health',
 'num__absences',
 'cat__school_GP',
 'cat__school_MS']

# Sauvegarde du split

In [29]:
ot.save_split(X_train, X_test, y_train, y_test)

# Vérification : on recharge et on compare les formes
Xtr, Xte_, ytr, yte = ot.load_split()
assert Xtr.shape == X_train.shape and Xte_.shape == X_test.shape
assert len(ytr) == len(y_train) and len(yte) == len(y_test)
print("Split sauvegardé dans", ot.SPLIT_DIR)

Split sauvegardé dans split


# Décisions pour la suite

* Cible : `G3`, en régression.
* Split 80 / 20 avec `random_state` fixé, stratifié sur `G3 == 0`, sauvegardé dans `split/`.
* Les 15 élèves avec `G3 = 0` sont conservés dans les fichiers ; la variante sans zéros pourra se faire dans `3-modeles` par filtrage.
* Trois scénarios de variables : avec `G1` et `G2` (`A_G1_G2`), puis seulement avec `G1` (`B_G1`) et enfin sans les notes (`C_sans_notes`).
* Prétraitement appris dans un `Pipeline` pendant la validation croisée, jamais sur le jeu complet.
* Le test set ne sera utilisé qu'une fois, dans `4-evaluation`.